# WallGo RL Training — Colab GPU Setup

**使用方法：** Runtime → Change runtime type → 选 T4 GPU → Run All

免费 T4 GPU 比 M1 MPS 快 3-5 倍做神经网络训练。

## 1. 检查 GPU 是否可用

In [ ]:
# 先确认 Colab 给了你一块 GPU
!nvidia-smi

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

## 2. 克隆项目 + 安装 Rust 引擎

In [ ]:
# ====== 改成你自己的 GitHub repo URL ======
REPO_URL = "https://github.com/YOUR_USERNAME/Wall-go-RL-Training.git"
# ==========================================

!git clone {REPO_URL} /content/wallgo
%cd /content/wallgo

In [ ]:
# 安装 Rust 工具链（大约 30 秒）
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = f"/root/.cargo/bin:{os.environ['PATH']}"
!rustc --version

In [ ]:
# 安装 Python 依赖
!pip install gymnasium stable-baselines3 sb3-contrib maturin rich

In [ ]:
# 编译 Rust 引擎（大约 1-2 分钟）
# Colab 没有 virtualenv，所以不能用 maturin develop
# 改用 maturin build 生成 .whl 文件，再 pip install 安装
!cd wallgo_rs && maturin build --release
!pip install wallgo_rs/target/wheels/*.whl

# 验证 Rust 引擎加载成功
!python -c "from wallgo_rs import WallGoEnv; print('Rust engine OK!')"

## 3. （可选）上传已有的 checkpoint 继续训练

如果你想从 M1 上训了一半的模型继续训练，把 checkpoint zip 文件上传到 `checkpoints/` 目录。

In [ ]:
# 方法 A：从 Google Drive 加载（推荐，不会因为断连丢失）
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/wallgo_checkpoints/*.zip /content/wallgo/checkpoints/

# 方法 B：手动上传（左侧文件面板拖拽）
!mkdir -p checkpoints
!ls -la checkpoints/

## 4. 开始训练！

In [ ]:
# 在 Colab 上，n-envs 可以开更大（Colab 有更多 CPU 核心）
# --steps: 总训练步数
# --n-envs: 并行游戏数（Colab 上可以开 8-16）
# --resume: 从最新 checkpoint 继续训练（如果有的话）

!python train.py --steps 5000000 --n-envs 8 --resume

## 5. 训练完保存结果

Colab 断连后文件会丢失！记得保存到 Google Drive。

In [ ]:
# 保存 checkpoints 到 Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/wallgo_checkpoints
!cp checkpoints/*.zip /content/drive/MyDrive/wallgo_checkpoints/
print("Checkpoints saved to Google Drive!")

## 6. 快速评估

In [ ]:
!python run_eval.py